In [1]:
#!pip install faker pymongo

In [2]:
from pymongo import MongoClient
import random
from faker import Faker
from datetime import datetime, timedelta
import uuid

fake = Faker('ru_RU')

client = MongoClient('mongodb://mongodb:27017/')
db = client['etl_database']

In [3]:
def generate_user_sessions(n=1000):
    sessions = []
    user_ids = [f"user_{i}" for i in range(1, 201)]  # 200 пользователей
    pages = ["/home", "/products"] + ["/products/{}".format(i) for i in range(1, 51)] + ["/cart", "/checkout", "/profile", "/search", "/about"]
    device_types = ["mobile", "desktop", "tablet"]
    browsers = ["Chrome", "Firefox", "Safari", "Edge"]
    os_list = ["iOS", "Android", "Windows", "macOS"]
    actions_list = ["login", "logout", "view_product", "add_to_cart", "remove_from_cart", "checkout", "search", "view_profile", "view_order"]
    
    for i in range(1, n+1):
        session_id = f"sess_{i:04d}"
        user_id = random.choice(user_ids)
        start_time = fake.date_time_between(start_date='-30d', end_date='now')
        end_time = start_time + timedelta(minutes=random.randint(1, 120))
        pages_visited = random.sample(pages, k=random.randint(1, 8))
        device = {
            "type": random.choice(device_types),
            "browser": random.choice(browsers),
            "os": random.choice(os_list)
        }
        actions = random.sample(actions_list, k=random.randint(0, 5))
        
        session = {
            "session_id": session_id,
            "user_id": user_id,
            "start_time": start_time.isoformat() + "Z",
            "end_time": end_time.isoformat() + "Z",
            "pages_visited": pages_visited,
            "device": device,
            "actions": actions
        }
        sessions.append(session)
    return sessions

In [4]:
def generate_event_logs(n=1000):
    events = []
    event_types = ["page_view", "click", "scroll", "purchase", "error", "login_attempt"]
    details_templates = [
        {"page": "/home", "element": "button"},
        {"page": "/products", "product_id": "prod_{}"},
        {"error_code": 404, "message": "Not Found"},
        {"amount": random.randint(10, 500), "currency": "RUB"},
        {"search_query": fake.word()}
    ]
    
    for i in range(1, n+1):
        event_id = f"evt_{i:06d}"
        timestamp = fake.date_time_between(start_date='-30d', end_date='now')
        event_type = random.choice(event_types)
        if event_type == "page_view":
            details = {"page": random.choice(["/home", "/products", "/cart", "/profile"])}
        elif event_type == "click":
            details = {"element": random.choice(["button", "link", "image"]), "page": random.choice(["/home", "/products"])}
        elif event_type == "purchase":
            details = {"order_id": f"ord_{random.randint(1000,9999)}", "amount": round(random.uniform(10, 500), 2)}
        elif event_type == "error":
            details = {"code": random.choice([404, 500, 403]), "message": fake.sentence()}
        elif event_type == "login_attempt":
            details = {"success": random.choice([True, False]), "method": "password"}
        else:
            details = {"info": fake.word()}
        
        event = {
            "event_id": event_id,
            "timestamp": timestamp.isoformat() + "Z",
            "event_type": event_type,
            "details": details
        }
        events.append(event)
    return events

In [5]:
def generate_support_tickets(n=1000):
    tickets = []
    statuses = ["open", "in_progress", "resolved", "closed"]
    issue_types = ["payment", "technical", "account", "product", "delivery"]
    user_ids = [f"user_{i}" for i in range(1, 201)]
    
    for i in range(1, n+1):
        ticket_id = f"ticket_{i:04d}"
        user_id = random.choice(user_ids)
        status = random.choice(statuses)
        issue_type = random.choice(issue_types)
        created_at = fake.date_time_between(start_date='-60d', end_date='-1d')
        # Сообщения
        num_messages = random.randint(1, 5)
        messages = []
        for j in range(num_messages):
            sender = "user" if j % 2 == 0 else "support"
            msg_time = created_at + timedelta(hours=random.randint(1, 48) * (j+1))
            msg_time = min(msg_time, datetime.now())
            message = {
                "sender": sender,
                "message": fake.sentence(nb_words=10),
                "timestamp": msg_time.isoformat() + "Z"
            }
            messages.append(message)
        updated_at = messages[-1]["timestamp"] if messages else created_at.isoformat() + "Z"
        
        ticket = {
            "ticket_id": ticket_id,
            "user_id": user_id,
            "status": status,
            "issue_type": issue_type,
            "messages": messages,
            "created_at": created_at.isoformat() + "Z",
            "updated_at": updated_at
        }
        tickets.append(ticket)
    return tickets

In [6]:
def generate_user_recommendations(n=1000):
    recs = []
    user_ids = [f"user_{i}" for i in range(1, 201)]
    product_ids = [f"prod_{i}" for i in range(101, 501)]
    
    for _ in range(n):
        user_id = random.choice(user_ids)
        recommended = random.sample(product_ids, k=random.randint(3, 10))
        last_updated = fake.date_time_between(start_date='-7d', end_date='now')
        rec = {
            "user_id": user_id,
            "recommended_products": recommended,
            "last_updated": last_updated.isoformat() + "Z"
        }
        recs.append(rec)
    return recs

In [7]:
def generate_moderation_queue(n=1000):
    queue = []
    user_ids = [f"user_{i}" for i in range(1, 201)]
    product_ids = [f"prod_{i}" for i in range(101, 501)]
    statuses = ["pending", "approved", "rejected"]
    flags_list = ["contains_images", "contains_profanity", "spam", "offensive", "promotional"]
    
    for i in range(1, n+1):
        review_id = f"rev_{i:04d}"
        user_id = random.choice(user_ids)
        product_id = random.choice(product_ids)
        review_text = fake.paragraph(nb_sentences=3)
        rating = random.randint(1, 5)
        moderation_status = random.choice(statuses)
        flags = random.sample(flags_list, k=random.randint(0, 3))
        submitted_at = fake.date_time_between(start_date='-30d', end_date='now')
        
        review = {
            "review_id": review_id,
            "user_id": user_id,
            "product_id": product_id,
            "review_text": review_text,
            "rating": rating,
            "moderation_status": moderation_status,
            "flags": flags,
            "submitted_at": submitted_at.isoformat() + "Z"
        }
        queue.append(review)
    return queue

In [8]:
# Генерация данных
print("Генерация UserSessions...")
user_sessions = generate_user_sessions(1000)
print("Генерация EventLogs...")
event_logs = generate_event_logs(1000)
print("Генерация SupportTickets...")
support_tickets = generate_support_tickets(1000)
print("Генерация UserRecommendations...")
user_recommendations = generate_user_recommendations(1000)
print("Генерация ModerationQueue...")
moderation_queue = generate_moderation_queue(1000)

# Вставка в MongoDB
db.user_sessions.insert_many(user_sessions)
db.event_logs.insert_many(event_logs)
db.support_tickets.insert_many(support_tickets)
db.user_recommendations.insert_many(user_recommendations)
db.moderation_queue.insert_many(moderation_queue)

print("Данные успешно вставлены!")

Генерация UserSessions...
Генерация EventLogs...
Генерация SupportTickets...
Генерация UserRecommendations...
Генерация ModerationQueue...
Данные успешно вставлены!


In [13]:
print("Количество документов в коллекциях:")
print("user_sessions:", db.user_sessions.count_documents({}))
print("event_logs:", db.event_logs.count_documents({}))
print("support_tickets:", db.support_tickets.count_documents({}))
print("user_recommendations:", db.user_recommendations.count_documents({}))
print("moderation_queue:", db.moderation_queue.count_documents({}))

Количество документов в коллекциях:
user_sessions: 0
event_logs: 0
support_tickets: 0
user_recommendations: 0
moderation_queue: 0


In [11]:
print("Пример документа из user_sessions:")
print(db.user_sessions.find_one())

print("\nПример из support_tickets (с сообщениями):")
print(db.support_tickets.find_one())

Пример документа из user_sessions:
{'_id': ObjectId('69aec281d5913d12049b63e9'), 'session_id': 'sess_0001', 'user_id': 'user_179', 'start_time': '2026-03-04T12:49:39.382160Z', 'end_time': '2026-03-04T12:51:39.382160Z', 'pages_visited': ['/products/23', '/products/17', '/products/33', '/products/42', '/products/48', '/profile'], 'device': {'type': 'tablet', 'browser': 'Edge', 'os': 'Windows'}, 'actions': ['view_product', 'checkout']}

Пример из support_tickets (с сообщениями):
{'_id': ObjectId('69aec281d5913d12049b6bb9'), 'ticket_id': 'ticket_0001', 'user_id': 'user_187', 'status': 'resolved', 'issue_type': 'account', 'messages': [{'sender': 'user', 'message': 'Желание секунда выражение сустав еврейский господь дыхание изображать бригада каюта.', 'timestamp': '2026-01-28T21:50:11.172731Z'}], 'created_at': '2026-01-27T11:50:11.172731Z', 'updated_at': '2026-01-28T21:50:11.172731Z'}
